# Randomized Motif Search

In [2]:
import os

def get_dataset_lines(filename):
    path_to_download_folder = os.path.join(os.path.expanduser('~'), 'Downloads', filename)
    with open(path_to_download_folder) as f:
        return f.read().strip().split('\n')

# Randomized Motif Search Problem
Implement RandomizedMotifSearch.

**Input**: Integers *k* and *t*, followed by a collection of strings *Dna*.

**Output**: A collection *BestMotifs* resulting from running RandomizedMotifSearch(*Dna*, *k*, *t*) 1,000 times. Remember to use pseudocounts!

**Code Challenge**: Implement RandomizedMotifSearch.

**Sample Input**:

```
8 5
CGCCCCTCTCGGGGGTGTTCAGTAAACGGCCA GGGCGAGGTATGTGTAAGTGCCAAGGTGCCAG TAGTACCGAGACCGAAAGAAGTATACAGGCGT TAGATCAAGTTTCAGGTGCACGTCGGTGAACC AATCCACCAGCTCCACGTGCAATGTTGGCCTA
```

**Sample Output**:

```
TCTCGGGG CCAAGGTG TACAGGCG TTCAGGTG TCCACGTG
```

In [3]:
# Helper functions from Module 3
def Count(Motifs):
    motif_length = len(Motifs[0])
    number_of_motifs = len(Motifs)
    
    counts = {}
    for nucleotide in "ACGT":
        counts[nucleotide] = [0] * motif_length
            
    for i in range(number_of_motifs):
        for j in range(motif_length):
            nucleotide = Motifs[i][j]
            counts[nucleotide][j] += 1
    return counts

def Score(Motifs):
    counts = Count(Motifs)
    motif_length = len(Motifs[0])
    number_of_motifs = len(Motifs)
    score = 0
    for j in range(motif_length):
        max_count = max(counts[base][j] for base in 'ACGT')
        score += (number_of_motifs - max_count)
    return score

def CountWithPseudocounts(Motifs):
    motif_length = len(Motifs[0])
    number_of_motifs = len(Motifs)
    
    counts = {}
    for nucleotide in "ACGT":
        counts[nucleotide] = [1] * motif_length
            
    for i in range(number_of_motifs):
        for j in range(motif_length):
            nucleotide = Motifs[i][j]
            counts[nucleotide][j] += 1
    return counts

def FormProfileWithPseudocounts(Motifs):
    counts = CountWithPseudocounts(Motifs)
    number_of_motifs = len(Motifs)
    profile = {}
    for base in 'ACGT':
        profile[base] = [count / (number_of_motifs + 4) for count in counts[base]]
    return profile

def Pr(Pattern, Profile):
    prob = 1
    for i in range(len(Pattern)):
        prob *= Profile[Pattern[i]][i]
    return prob

def ProfileMostProbableKmer(Text, k, Profile):
    max_prob = -1
    most_probable = Text[0:k]
    
    for i in range(len(Text) - k + 1):
        Pattern = Text[i:i+k]
        prob = Pr(Pattern, Profile)
        if prob > max_prob:
            max_prob = prob
            most_probable = Pattern
            
    return most_probable

In [4]:
import random

def Motifs(Profile, Dna):
    k = len(Profile['A'])
    return [ProfileMostProbableKmer(seq, k, Profile) for seq in Dna]

def RandomizedMotifSearch(Dna, k, t):
    # Randomly select k-mers from each string in Dna
    M = [random.randint(0, len(Dna[i]) - k) for i in range(t)]
    BestMotifs = [Dna[i][r:r+k] for i, r in enumerate(M)]
    
    while True:
        Profile = FormProfileWithPseudocounts(BestMotifs)
        CurrentMotifs = Motifs(Profile, Dna)
        if Score(CurrentMotifs) < Score(BestMotifs):
            BestMotifs = CurrentMotifs
        else:
            return BestMotifs

def RepeatedRandomizedMotifSearch(Dna, k, t, N=1000):
    BestMotifs = RandomizedMotifSearch(Dna, k, t)
    MinScore = Score(BestMotifs)
    
    for _ in range(N - 1):
        CurrentMotifs = RandomizedMotifSearch(Dna, k, t)
        CurrentScore = Score(CurrentMotifs)
        if CurrentScore < MinScore:
            BestMotifs = CurrentMotifs
            MinScore = CurrentScore
            
    return BestMotifs

In [5]:
# Sample Input
k = 8
t = 5
Dna = [
    "CGCCCCTCTCGGGGGTGTTCAGTAAACGGCCA",
    "GGGCGAGGTATGTGTAAGTGCCAAGGTGCCAG",
    "TAGTACCGAGACCGAAAGAAGTATACAGGCGT",
    "TAGATCAAGTTTCAGGTGCACGTCGGTGAACC",
    "AATCCACCAGCTCCACGTGCAATGTTGGCCTA"
]

# Run the algorithm 1000 times
best_motifs = RepeatedRandomizedMotifSearch(Dna, k, t, 1000)
print(*best_motifs)

# Test Assertion
expected_output = ['TCTCGGGG', 'CCAAGGTG', 'TACAGGCG', 'TTCAGGTG', 'TCCACGTG']
assert best_motifs == expected_output, f"Expected {expected_output}, but got {best_motifs}"
print("Test passed!")

TCTCGGGG CCAAGGTG TACAGGCG TTCAGGTG TCCACGTG
Test passed!


In [6]:
# Test Dataset
test_dataset_filename = 'dataset_30307_5.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    k, t = map(int, lines[0].split())
    Dna = lines[1].split()
    
    best_motifs = RepeatedRandomizedMotifSearch(Dna, k, t, 1000)
    print(*best_motifs)
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

TGTCGGCACGCCTGT AGATCTCTTTCCCGC AGACGTGCTTCCCGC AGACGGCTTTGTGGC AGACGGCTTTCATTC AGACGGCTTTCCGCA AGAAAACTTTCCCGC AGACGGCCGCCCCGC AGACGGGACTCCCGC AGACCTTTTTCCCGC AGACGGCTCGACCGC CTCCGGCTTTCCCGC AGACGGCTTCTACGC AGACGGTGCTCCCGC AATAGGCTTTCCCGC AGACGCGCTTCCCGC GGACGGCTTTCCCAT AGACCAGTTTCCCGC AGTATGCTTTCCCGC CAACGGCTTTCCCGA


**Exercise Break**: Compute the probability that ten randomly selected 15-mers from the ten 600-nucleotide long strings in the Subtle Motif Problem capture at least one implanted 15-mer. (Allowable error: 0.000001)

In [7]:
N = 600
k = 15
t = 10

# Number of possible k-mers in a sequence of length N
num_kmers = N - k + 1

# Probability of picking the implanted motif in one sequence
prob_one = 1 / num_kmers

# Probability of NOT picking the implanted motif in one sequence
prob_not_one = 1 - prob_one

# Probability of NOT picking the implanted motif in ANY of the t sequences
prob_not_any = prob_not_one ** t

# Probability of picking at least one implanted motif
prob_at_least_one = 1 - prob_not_any

print(f"Probability: {prob_at_least_one}")

Probability: 0.01693439692911025


**Exercise Break**: Compute the probability that ten randomly selected 15-mers from ten 600-nucleotide long strings (as in the Subtle Motif Problem) capture at least two implanted 15-mers. (Allowable error: 0.000001)

In [8]:
N = 600
k = 15
t = 10

# Number of possible k-mers in a sequence of length N
num_kmers = N - k + 1

# Probability of picking the implanted motif in one sequence
p = 1 / num_kmers
q = 1 - p

# Probability of capturing exactly 0 implanted motifs
prob_0 = q ** t

# Probability of capturing exactly 1 implanted motif
prob_1 = t * p * (q ** (t - 1))

# Probability of capturing at least 2 implanted motifs
prob_at_least_2 = 1 - (prob_0 + prob_1)

print(f"Probability of capturing at least 2 implanted motifs: {prob_at_least_2}")

Probability of capturing at least 2 implanted motifs: 0.00012985670567622343


# Gibbs Sampler Problem
Implement GibbsSampler.

**Input**: Integers *k*, *t*, and *N*, followed by a collection of strings *Dna*.

**Output**: The strings *BestMotifs* resulting from running GibbsSampler(*Dna*, *k*, *t*, *N*) with 20 random starts. Remember to use pseudocounts!

**Code Challenge**: Implement GibbsSampler.

**Sample Input**:

```
8 5 100
CGCCCCTCTCGGGGGTGTTCAGTAACCGGCCA GGGCGAGGTATGTGTAAGTGCCAAGGTGCCAG TAGTACCGAGACCGAAAGAAGTATACAGGCGT TAGATCAAGTTTCAGGTGCACGTCGGTGAACC AATCCACCAGCTCCACGTGCAATGTTGGCCTA
```

**Sample Output**:

```
TCTCGGGG CCAAGGTG TACAGGCG TTCAGGTG TCCACGTG
```

In [9]:
def ProfileRandomlyGeneratedKmer(Text, k, Profile):
    probabilities = []
    for i in range(len(Text) - k + 1):
        Pattern = Text[i:i+k]
        probabilities.append(Pr(Pattern, Profile))
    
    # Normalize probabilities
    total_prob = sum(probabilities)
    if total_prob == 0:
        # If all probabilities are 0, pick uniformly
        probabilities = [1/len(probabilities)] * len(probabilities)
    else:
        probabilities = [p / total_prob for p in probabilities]
        
    # Randomly select a k-mer index based on probabilities
    # random.choices returns a list, so we take the first element
    index = random.choices(range(len(probabilities)), weights=probabilities, k=1)[0]
    
    return Text[index:index+k]

def GibbsSampler(Dna, k, t, N):
    # Randomly select k-mers from each string in Dna
    M = [random.randint(0, len(Dna[i]) - k) for i in range(t)]
    Motifs = [Dna[i][r:r+k] for i, r in enumerate(M)]
    BestMotifs = Motifs[:]
    
    for j in range(N):
        i = random.randint(0, t - 1)
        
        # Create profile from all motifs except the i-th one
        Motifs_excluding_i = Motifs[:i] + Motifs[i+1:]
        Profile = FormProfileWithPseudocounts(Motifs_excluding_i)
        
        # Generate new motif for the i-th sequence
        Motifs[i] = ProfileRandomlyGeneratedKmer(Dna[i], k, Profile)
        
        if Score(Motifs) < Score(BestMotifs):
            BestMotifs = Motifs[:]
            
    return BestMotifs

def RepeatedGibbsSampler(Dna, k, t, N, num_starts=20):
    BestMotifs = GibbsSampler(Dna, k, t, N)
    MinScore = Score(BestMotifs)
    
    for _ in range(num_starts - 1):
        CurrentMotifs = GibbsSampler(Dna, k, t, N)
        CurrentScore = Score(CurrentMotifs)
        if CurrentScore < MinScore:
            BestMotifs = CurrentMotifs
            MinScore = CurrentScore
            
    return BestMotifs

In [12]:
# Sample Input for GibbsSampler
k = 8
t = 5
N = 100
Dna = [
    "CGCCCCTCTCGGGGGTGTTCAGTAACCGGCCA",
    "GGGCGAGGTATGTGTAAGTGCCAAGGTGCCAG",
    "TAGTACCGAGACCGAAAGAAGTATACAGGCGT",
    "TAGATCAAGTTTCAGGTGCACGTCGGTGAACC",
    "AATCCACCAGCTCCACGTGCAATGTTGGCCTA"
]

# Run the algorithm with 20 random starts
best_motifs = RepeatedGibbsSampler(Dna, k, t, N, 20)
print(*best_motifs)

# Test Assertion
expected_output = ['TCTCGGGG', 'CCAAGGTG', 'TACAGGCG', 'TTCAGGTG', 'TCCACGTG']
# Note: GibbsSampler is randomized, so it might not always converge to the global optimum with few iterations/starts.
# But with these parameters it should be consistent enough for this small example.
assert best_motifs == expected_output, f"Expected {expected_output}, but got {best_motifs}"
print("Test passed!")

TCTCGGGG CCAAGGTG TACAGGCG TTCAGGTG TCCACGTG
Test passed!


In [11]:
# Test Dataset
test_dataset_filename = 'dataset_30309_11.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    k, t, N = map(int, lines[0].split())
    Dna = lines[1].split()
    
    # Usually for the dataset we might need more starts or iterations
    best_motifs = RepeatedGibbsSampler(Dna, k, t, N, 20)
    print(*best_motifs)
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

AACCCGCATGGCTGT GCCGAAGATGGCTGT GCCTTGGAGCACTGT GCAGCGGATGGCTGT GCCTATTATGGCTGT GTGGTGGATGGCTGT GCCGGCGATGGCTGT GCCTTGGATGGTGTT GCCTTGGATATATGT GCCTTCCTTGGCTGT GCCTTGGGCAGCTGT GCCTTTTCTGGCTGT GCCTGCTATGGCTGT GCCTTGGATGGCACA CCCTTGGATGGCTTG AAGTTGGATGGCTGT GCCTTGATAGGCTGT AGCTTGGATGGCTGG GCCTTGGATGCAGGT GCCTTGTCGGGCTGT


# Final Challenge
Infer the profile of the DosR motif and find all its putative occurrences in *Mycobacterium tuberculosis*.

In [14]:
# Final Challenge
# Read DosR dataset
with open('DosR.txt', 'r') as f:
    DosR_Dna = [line.strip() for line in f.readlines()]

print(f"Number of sequences: {len(DosR_Dna)}")
print(f"Length of first sequence: {len(DosR_Dna[0])}")

# Run RepeatedGibbsSampler
# We'll assume k=20 based on typical DosR motif length in this course context
k = 20
t = len(DosR_Dna)
N = 200 # Iterations for GibbsSampler
num_starts = 20 # Number of random starts

best_motifs_dosr = RepeatedGibbsSampler(DosR_Dna, k, t, N, num_starts)

print("Best Motifs found:")
for motif in best_motifs_dosr:
    print(motif)

print(f"Score: {Score(best_motifs_dosr)}")

def Consensus(Motifs):
    k = len(Motifs[0])
    count = Count(Motifs)
    consensus = ""
    for j in range(k):
        m = 0
        frequentSymbol = ""
        for symbol in "ACGT":
            if count[symbol][j] > m:
                m = count[symbol][j]
                frequentSymbol = symbol
        consensus += frequentSymbol
    return consensus

print(f"Consensus Motif: {Consensus(best_motifs_dosr)}")

# Print Profile
profile = FormProfileWithPseudocounts(best_motifs_dosr)
print("\nProfile Matrix (with pseudocounts):")
for base in 'ACGT':
    print(f"{base}: {['{:.2f}'.format(x) for x in profile[base]]}")

Number of sequences: 10
Length of first sequence: 250
Best Motifs found:
CGGGACTTCAGGCCCTATCG
CGGGTCAAACGACCCTAGTG
CGGGACGTAAGTCCCTAACG
GGCCGTCTCAGTACCCAGCC
CGTGACCGACGTCCCCAGCC
GAGGACCTTCGGCCCCACCC
GGGGACTTCTGTCCCTAGCC
TGGGACTTTCGGCCCTGTCC
GGGGACCAACGCCCCTGGGA
GGGGACCGAAGTCCCCGGGC
Score: 55
Consensus Motif: GGGGACCTACGTCCCTAGCC

Profile Matrix (with pseudocounts):
A: ['0.07', '0.14', '0.07', '0.07', '0.64', '0.07', '0.14', '0.21', '0.43', '0.36', '0.07', '0.14', '0.14', '0.07', '0.07', '0.07', '0.57', '0.14', '0.07', '0.14']
C: ['0.36', '0.07', '0.14', '0.14', '0.07', '0.71', '0.43', '0.07', '0.29', '0.43', '0.07', '0.14', '0.71', '0.79', '0.79', '0.36', '0.07', '0.14', '0.57', '0.50']
G: ['0.43', '0.71', '0.64', '0.71', '0.14', '0.07', '0.14', '0.21', '0.07', '0.07', '0.79', '0.29', '0.07', '0.07', '0.07', '0.07', '0.29', '0.50', '0.21', '0.29']
T: ['0.14', '0.07', '0.14', '0.07', '0.14', '0.14', '0.29', '0.50', '0.21', '0.14', '0.07', '0.43', '0.07', '0.07', '0.07', '0.50', '0.07', 